In [2]:
#module1
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.3/112.6 GB disk)


In [3]:
!pip install -q pypdf sentence-transformers
# nếu bạn dùng FAISS:
!pip install -q faiss-cpu  || echo "FAISS install failed → sẽ dùng numpy fallback"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.6/326.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 76.9 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#copy link bên roboflow

In [ ]:
# Train YOLO11n on COCO8 for 3 epochs
!yolo train model=yolo11n.pt data=/content/My-Traffic-Train-3/data.yaml epochs=100 imgsz=640

In [ ]:
# ================== MODULE 1 (YOLOv11): Keyframe -> Detect -> JSON/CSV ==================
# ==== 0) IMPORTS ====
import os, re, json, csv, gc, time
from pathlib import Path
from typing import List, Dict, Any, Tuple
import numpy as np
import cv2
from ultralytics import YOLO
try:
    import torch
except Exception:
    torch = None

# ==== 1) KHAI BÁO ĐƯỜNG DẪN & THAM SỐ (SỬA Ở ĐÂY) ====
WEIGHTS            = "/content/data/best.pt"       # hoặc "yolo11s.pt"
VIDEO_IN           = "/content/drive/MyDrive/videos"              # file .mp4 hoặc folder video
OUT_DIR            = "/content/runs_meta"
META_JSON          = f"{OUT_DIR}/metadata.json"
DET_CSV            = f"{OUT_DIR}/detections.csv"

# Keyframe
KEYFRAME_MODE      = "time"   # "time" hoặc "scene"
SECONDS_PER_SAMPLE = 1.0
MAX_FRAMES_PER_VID = 60       # 0 = không giới hạn
SCENE_THRESHOLD    = 0.40

# YOLO
CONF_THRES         = 0.25
IOU_THRES          = 0.45
if (torch and torch.cuda.is_available()) or os.path.exists("/proc/driver/nvidia/version"):
    DEVICE = 0
else:
    DEVICE = "cpu"
BATCH              = 8

# ==== 2) TIỆN ÍCH ====
def ensure_dir(p: str | Path): Path(p).mkdir(parents=True, exist_ok=True)
def norm_vpath(p: str) -> str:
    if not p: return ""
    return re.sub(r"/{2,}", "/", p.replace("\\", "/").strip())

def get_video_info(vid_path: str | Path) -> Tuple[float, float, int]:
    cap = cv2.VideoCapture(str(vid_path))
    if not cap.isOpened(): return 0.0, 0.0, 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = frames / fps if fps > 0 else 0.0
    cap.release()
    return fps, duration, frames

def list_videos(input_path: str | Path) -> List[Path]:
    p = Path(input_path)
    if p.is_file(): return [p]
    exts = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}
    return sorted([q for q in p.rglob("*") if q.suffix.lower() in exts])

# ==== 3) KEYFRAME ====
def sample_timestamps_by_time(duration: float, step: float, limit: int) -> List[float]:
    if duration <= 0: return []
    ts = [round(t, 3) for t in np.arange(0.0, duration, step).tolist()]
    return ts[:limit] if limit > 0 else ts

def sample_timestamps_by_scene(vid_path: str | Path, thr: float, limit: int) -> List[float]:
    cap = cv2.VideoCapture(str(vid_path))
    if not cap.isOpened(): return []
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    if fps <= 0: cap.release(); return []
    prev_hist, timestamps, idx = None, [0.0], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hist = cv2.normalize(cv2.calcHist([gray],[0],None,[64],[0,256]), None).flatten()
        if prev_hist is not None:
            diff = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_BHATTACHARYYA)  # 0~1
            if diff >= thr:
                t = idx / fps
                if not timestamps or (t - timestamps[-1] >= 0.25):
                    timestamps.append(round(t, 3))
                    if limit > 0 and len(timestamps) >= limit: break
        prev_hist, idx = hist, idx + 1
    cap.release()
    return timestamps

def read_frame_at(vid_path: str | Path, t_sec: float):
    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    if fps <= 0: cap.release(); return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(t_sec * fps))
    ok, frame = cap.read(); cap.release()
    return frame if ok else None

def extract_keyframes(vid_path: Path) -> Tuple[List[float], List[np.ndarray]]:
    _, duration, _ = get_video_info(vid_path)
    if KEYFRAME_MODE == "scene":
        ts = sample_timestamps_by_scene(vid_path, SCENE_THRESHOLD, MAX_FRAMES_PER_VID)
    else:
        ts = sample_timestamps_by_time(duration, SECONDS_PER_SAMPLE, MAX_FRAMES_PER_VID)
    imgs = [read_frame_at(vid_path, t) for t in ts]
    return ts, imgs

# ==== 4) YOLOv11 INFERENCE ====
def auto_find_best_weights(root="runs/detect"):
    rootp = Path(root)
    if not rootp.exists(): return None
    cands = list(rootp.glob("**/weights/best.pt"))
    if not cands: return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(cands[0])

def init_model(weights: str | None) -> YOLO:
    if not weights:
        w = auto_find_best_weights()
        weights = w if w else "yolo11s.pt"
        print(f"[INFO] dùng weights: {weights}")
    else:
        if Path(weights).suffix.lower() == ".pt":
            assert Path(weights).exists(), f"Không thấy weights: {weights}"
    print(f"[YOLOv11] Loading: {weights}")
    return YOLO(weights)

def run_yolo_on_frames(model: YOLO, frames: List[np.ndarray], conf=0.25, iou=0.45, device="cpu", batch=8):
    results = [None] * len(frames)
    idxs = [i for i,f in enumerate(frames) if f is not None]
    if not idxs: return results
    for k in range(0, len(idxs), batch):
        bi = idxs[k:k+batch]
        imgs = [frames[i] for i in bi]
        r = model.predict(source=imgs, conf=conf, iou=iou, device=device, verbose=False)
        for j, ridx in enumerate(bi): results[ridx] = r[j]
    return results

def postprocess_results(result, names: Dict[int,str]) -> List[Dict[str,Any]]:
    if result is None or not hasattr(result, "boxes"): return []
    boxes = result.boxes
    if boxes is None: return []
    xyxy = boxes.xyxy.cpu().numpy() if hasattr(boxes, "xyxy") else []
    cls  = boxes.cls.cpu().numpy().astype(int) if hasattr(boxes, "cls") else []
    conf = boxes.conf.cpu().numpy() if hasattr(boxes, "conf") else []
    out = []
    for i in range(len(cls)):
        x1,y1,x2,y2 = map(float, xyxy[i]); c = int(cls[i])
        out.append({"cls": c, "name": names.get(c, str(c)), "conf": float(conf[i]), "xyxy": [x1,y1,x2,y2]})
    return out

# ==== 5) PIPELINE & CHẠY ====
def pipeline_module1_to_files(videos: List[Path], weights: str | None, meta_json: str, det_csv: str):
    ensure_dir(Path(meta_json).parent)
    model = init_model(weights)
    names = getattr(model.model, "names", None) or getattr(model, "names", {}) or {}

    all_meta = []
    with open(det_csv, "w", encoding="utf-8", newline="") as fcsv:
        wr = csv.DictWriter(fcsv, fieldnames=["video_path","t","cls","name","conf","x1","y1","x2","y2"])
        wr.writeheader()

        for i, vp in enumerate(list_videos(VIDEO_IN), 1):
            vpath = norm_vpath(str(vp))
            fps, duration, frames = get_video_info(vp)
            print(f"[{i}] {vp.name} | fps={fps:.2f} dur={duration:.1f}s frames={frames}")

            ts, imgs = extract_keyframes(vp)
            print(f"  - keyframes: {len(ts)}")

            yres = run_yolo_on_frames(model, imgs, conf=CONF_THRES, iou=IOU_THRES, device=DEVICE, batch=BATCH)

            kf = []
            for j, t in enumerate(ts):
                objs = postprocess_results(yres[j], names)
                kf.append({"t": float(t), "objects": objs})
                for o in objs:
                    wr.writerow({
                        "video_path": vpath, "t": float(t), "cls": o["cls"], "name": o["name"], "conf": o["conf"],
                        "x1": o["xyxy"][0], "y1": o["xyxy"][1], "x2": o["xyxy"][2], "y2": o["xyxy"][3],
                    })

            all_meta.append({
                "video_path": vpath, "fps": float(fps), "duration": float(duration),
                "frames": int(frames), "keyframes": kf
            })

            del imgs, yres, kf
            gc.collect()

    with open(meta_json, "w", encoding="utf-8") as f:
        json.dump({"created_at": time.time(), "video_count": len(all_meta), "items": all_meta},
                  f, ensure_ascii=False, indent=2)
    print(f"[OK] Saved metadata:   {meta_json}")
    print(f"[OK] Saved detections: {det_csv}")

# ==== RUN ====
assert Path(VIDEO_IN).exists(), f"Không thấy VIDEO_IN: {VIDEO_IN}"
ensure_dir(OUT_DIR)
pipeline_module1_to_files(list_videos(VIDEO_IN), WEIGHTS, META_JSON, DET_CSV)

# Preview nhanh
try:
    data = json.load(open(META_JSON, "r", encoding="utf-8"))
    print("Preview item 1:\n", json.dumps(data["items"][0], ensure_ascii=False, indent=2)[:1000])
except Exception as e:
    print("Preview lỗi:", e)


Module 2


In [4]:
# ==== MODULE 2 - COMMON SETUP ====
# Chỉnh ở đây để khớp dự án của bạn
RB_ROOT   = "roadbuddy"
FRAMES_DIR= f"{RB_ROOT}/data/frames"
RUNTIME_DIR=f"{RB_ROOT}/runtime"
KB_DIR    = f"{RB_ROOT}/kb"
PDF_DIR   = f"{KB_DIR}/pdfs"

SCENE_CACHE_PATH = f"{RUNTIME_DIR}/scene_cache.jsonl"        # từ bước detect (YOLO) đã chạy
SCENE_DESC_PATH  = f"{RUNTIME_DIR}/scene_description.jsonl"  # output M2.1

# VLM & Instruct settings
VLM_MODEL = "Salesforce/instructblip-vicuna-7b"  # có thể đổi sang flan-t5-xl nếu GPU yếu
FPS       = 15    # FPS tương ứng khi bạn trích frames
WIN_SEC   = 3     # độ dài cửa sổ mô tả cảnh
CONF_TH   = 0.35  # ngưỡng confidence

# Class names (đổi theo training của bạn)
SIGN_NAMES  = ["speed_limit","no_left","no_right","stop","priority","give_way","end_speed","other_sign"]
LIGHT_NAMES = ["red","yellow","green","green_left","green_straight","red_yellow","off"]

# Embedding model cho RAG
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RAG_OUT   = f"{KB_DIR}/index"
CHUNKS_JSONL = f"{RAG_OUT}/chunks.jsonl"
FAISS_INDEX  = f"{RAG_OUT}/faiss.index"

# ===== Imports =====
import os, json, re, gzip, glob, math, shutil
from pathlib import Path
from collections import defaultdict, Counter
from typing import List, Dict, Any, Iterable


In [5]:
# ==== FULL CODE: Scene Description + Spatial-Temporal Features ====
import os, json, math
from pathlib import Path
from typing import Dict, List, Any, Iterable
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

# ==================== CẤU HÌNH ====================
CSV_IN          = "/content/runs_meta/detections.csv"
LABEL_JSON      = "/content/data/label.json"  # ← File mapping labels của bạn
RB_ROOT         = "roadbuddy"
OUT_JSONL       = f"{RB_ROOT}/runtime/scene_cache.jsonl"
SCENE_DESC      = f"{RB_ROOT}/runtime/scene_description.jsonl"

FPS_FALLBACK    = 30.0
WIN_SEC         = 2.0   # Cửa sổ phân tích
CONF_TH         = 0.5   # Ngưỡng confidence

# Từ khóa nhận diện đèn giao thông
LIGHT_NAME_HINTS = {"den", "denDo", "denXanh", "denVang", "den_muiTen", "traffic_light"}

# ==================== ĐỌC METADATA LABELS ====================
def load_label_mapping(path: str) -> Dict[str, str]:
    """Đọc file label.json để map tên class"""
    if not os.path.isfile(path):
        print(f"⚠️ Không tìm thấy {path}, dùng tên gốc từ CSV")
        return {}

    with open(path, "r", encoding="utf-8") as f:
        labels = json.load(f)

    # Tạo reverse mapping: key -> description
    mapping = {}
    for key, desc in labels.items():
        mapping[key] = desc
        mapping[key.lower()] = desc  # thêm lowercase để match dễ hơn

    print(f"✅ Đã load {len(labels)} labels từ {path}")
    return mapping

LABEL_MAP = load_label_mapping(LABEL_JSON)

def get_label_name(raw_name: str) -> str:
    """Lấy tên chuẩn hóa từ label.json hoặc giữ nguyên"""
    return LABEL_MAP.get(raw_name, LABEL_MAP.get(raw_name.lower(), raw_name))

# ==================== PHÂN LOẠI ĐÈN/BIỂN ====================
def is_light(name: str) -> bool:
    """Kiểm tra xem có phải đèn giao thông không"""
    n = (str(name) or "").lower()
    return any(h.lower() in n for h in LIGHT_NAME_HINTS)

# ==================== CHUYỂN ĐỔI CSV -> SCENE_CACHE ====================
def csv_to_scene_cache():
    """Convert detections.csv -> scene_cache.jsonl"""
    df = pd.read_csv(CSV_IN)

    # Kiểm tra cột bắt buộc
    required = {"video_path", "name", "cls", "conf"}
    if not required.issubset(df.columns):
        raise RuntimeError(f"CSV thiếu cột: {required - set(df.columns)}")

    # Xử lý thời gian
    has_t = "t" in df.columns
    has_frame = "frame" in df.columns
    has_fps = "fps" in df.columns

    if not (has_t or has_frame):
        raise RuntimeError("CSV cần có cột 't' (giây) hoặc 'frame'")

    if not has_fps:
        df["fps"] = FPS_FALLBACK

    if not has_frame:
        df["frame"] = (df["t"] * df["fps"]).round().astype(int)
    if not has_t:
        df["t"] = df["frame"] / df["fps"]

    # Xử lý bbox
    if {"x1","y1","x2","y2"}.issubset(df.columns):
        pass
    elif "xyxy" in df.columns:
        def parse_bbox(s):
            try:
                s = str(s).replace("[","").replace("]","").replace(","," ")
                vals = [float(v) for v in s.split()]
                return vals if len(vals)==4 else [0,0,0,0]
            except:
                return [0,0,0,0]

        tmp = df["xyxy"].apply(parse_bbox)
        df["x1"] = tmp.apply(lambda v: v[0])
        df["y1"] = tmp.apply(lambda v: v[1])
        df["x2"] = tmp.apply(lambda v: v[2])
        df["y2"] = tmp.apply(lambda v: v[3])
    else:
        for c in ["x1","y1","x2","y2"]:
            df[c] = 0.0

    # Ghi scene_cache
    Path(f"{RB_ROOT}/runtime").mkdir(parents=True, exist_ok=True)

    count = 0
    with open(OUT_JSONL, "w", encoding="utf-8") as out:
        for (vpath, frame), g in df.groupby(["video_path", "frame"]):
            vid = os.path.splitext(os.path.basename(str(vpath)))[0] or "vid"
            rec = {"vid": vid, "frame": int(frame), "signs": [], "lights": [], "lanes": [], "ocr": []}

            for r in g.itertuples(index=False):
                entry = {
                    "cls": int(getattr(r, "cls")),
                    "conf": float(getattr(r, "conf")),
                    "name": str(getattr(r, "name", "")),  # giữ tên gốc
                    "bbox": [float(getattr(r,"x1",0)), float(getattr(r,"y1",0)),
                            float(getattr(r,"x2",0)), float(getattr(r,"y2",0))]
                }
                name = getattr(r, "name", "")
                (rec["lights"] if is_light(name) else rec["signs"]).append(entry)

            out.write(json.dumps(rec, ensure_ascii=False) + "\n")
            count += 1

    print(f"✅ Wrote {count} frames -> {OUT_JSONL}")

# ==================== ĐỌC SCENE_CACHE ====================
def _open_any(path: str):
    import gzip
    return gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, "r", encoding="utf-8")

def _load_scene_cache(path: str) -> Iterable[Dict[str, Any]]:
    with _open_any(path) as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception as e:
                print(f"[WARN] Bỏ dòng {ln}: {e}")
                continue
            if "vid" not in rec or "frame" not in rec:
                print(f"[WARN] Bỏ dòng {ln}: thiếu vid/frame")
                continue
            rec.setdefault("signs", [])
            rec.setdefault("lights", [])
            rec.setdefault("lanes", [])
            rec.setdefault("ocr", [])
            yield rec

# ==================== TRÍCH XUẤT ĐẶC TRƯNG SPATIAL-TEMPORAL ====================
def extract_spatiotemporal_features(records: List[Dict], fps: float) -> Dict:
    """
    Phân tích không gian-thời gian:
    - Vị trí & kích thước bbox theo thời gian
    - Vận tốc di chuyển (tracking)
    - Mật độ không gian
    """
    features = {
        "trajectories": [],
        "spatial_density": 0,
        "avg_movement_speed": 0
    }

    # Track objects qua các frame
    tracks = defaultdict(list)  # {name: [(frame, cx, cy, area), ...]}

    for r in records:
        frame = int(r.get("frame", 0))

        for obj_type in ["signs", "lights"]:
            for obj in r.get(obj_type, []):
                bbox = obj.get("bbox", [0,0,0,0])
                if len(bbox) == 4:
                    x1, y1, x2, y2 = bbox
                    cx = (x1 + x2) / 2
                    cy = (y1 + y2) / 2
                    area = max(0, (x2 - x1) * (y2 - y1))

                    name = obj.get("name", f"obj_{obj.get('cls')}")
                    tracks[name].append((frame, cx, cy, area))

    # Tính vận tốc
    all_speeds = []
    for name, track in tracks.items():
        if len(track) >= 2:
            track.sort(key=lambda x: x[0])  # sort theo frame
            speeds = []

            for i in range(1, len(track)):
                f1, x1, y1, a1 = track[i-1]
                f2, x2, y2, a2 = track[i]

                dt = (f2 - f1) / fps
                if dt > 0:
                    dist = math.sqrt((x2-x1)**2 + (y2-y1)**2)
                    speed = dist / dt  # pixels/second
                    speeds.append(speed)

            if speeds:
                features["trajectories"].append({
                    "name": name,
                    "avg_speed": float(np.mean(speeds)),
                    "max_speed": float(np.max(speeds)),
                    "frames": len(track)
                })
                all_speeds.extend(speeds)

    if all_speeds:
        features["avg_movement_speed"] = float(np.mean(all_speeds))

    # Mật độ không gian
    all_positions = []
    for track in tracks.values():
        all_positions.extend([(x, y) for _, x, y, _ in track])

    if len(all_positions) > 1:
        positions_arr = np.array(all_positions)
        features["spatial_density"] = float(np.std(positions_arr))

    return features

# ==================== TẠO MÔ TẢ SCENE ====================
def _summarize_window(records: List[Dict], conf_thres: float, fps: float) -> str:
    """Tạo mô tả ngắn gọn cho cửa sổ thời gian"""
    signs, lights, lane_flags, ocr_vals = Counter(), Counter(), Counter(), Counter()

    for r in records:
        # Biển báo
        for s in r.get("signs", []):
            try:
                if float(s.get("conf", 0)) >= conf_thres:
                    raw_name = s.get("name", f"sign_{s.get('cls')}")
                    label = get_label_name(raw_name)
                    signs[label] += 1
            except:
                pass

        # Đèn giao thông
        for l in r.get("lights", []):
            try:
                if float(l.get("conf", 0)) >= conf_thres:
                    raw_name = l.get("name", f"light_{l.get('cls')}")
                    label = get_label_name(raw_name)
                    lights[label] += 1
            except:
                pass

        # Làn đường
        for ln in r.get("lanes", []):
            t = ln.get("type")
            if t:
                lane_flags[t] += 1

        # OCR
        for o in r.get("ocr", []):
            txt = str(o.get("text", "")).strip()
            if txt:
                ocr_vals[txt] += 1

    # Tạo text summary
    parts = []
    if signs:
        parts.append("Biển: " + ", ".join(f"{k}×{v}" for k,v in signs.most_common(5)))
    if lights:
        parts.append("Đèn: " + ", ".join(f"{k}×{v}" for k,v in lights.most_common(3)))
    if lane_flags:
        parts.append("Làn: " + ", ".join(f"{k}×{v}" for k,v in lane_flags.most_common(3)))
    if ocr_vals:
        parts.append("OCR: " + ", ".join(f"'{k}'×{v}" for k,v in ocr_vals.most_common(3)))

    # Thêm phân tích spatial-temporal
    st_feat = extract_spatiotemporal_features(records, fps)

    if st_feat["avg_movement_speed"] > 0:
        parts.append(f"[ST: Chuyển động TB {st_feat['avg_movement_speed']:.1f}px/s]")

    if st_feat["trajectories"]:
        n_moving = len([t for t in st_feat["trajectories"] if t["avg_speed"] > 5])
        if n_moving > 0:
            parts.append(f"[{n_moving} đối tượng di chuyển]")

    return ". ".join(parts) + "." if parts else "Không có đặc trưng nổi bật."

# ==================== XÂY DỰNG SCENE DESCRIPTIONS ====================
def build_scene_descriptions():
    """Tạo file scene_description.jsonl với spatial-temporal features"""
    if not os.path.isfile(OUT_JSONL):
        raise FileNotFoundError(f"Không tìm thấy {OUT_JSONL}. Chạy csv_to_scene_cache() trước.")

    # Group theo video
    by_vid = defaultdict(list)
    total = 0
    for rec in _load_scene_cache(OUT_JSONL):
        by_vid[rec["vid"]].append(rec)
        total += 1

    if total == 0:
        raise RuntimeError("scene_cache rỗng")

    print(f"📊 Loaded {total} frames từ {len(by_vid)} videos")

    # Sort theo frame
    for vid in by_vid:
        by_vid[vid].sort(key=lambda r: int(r["frame"]))

    # Tạo descriptions
    win_frames = max(1, int(FPS_FALLBACK * WIN_SEC))
    wrote = 0

    with open(SCENE_DESC, "w", encoding="utf-8") as out:
        for vid, recs in by_vid.items():
            i, n = 0, len(recs)
            while i < n:
                j = min(i + win_frames, n)
                window = recs[i:j]

                t0 = round(int(window[0]["frame"]) / FPS_FALLBACK, 3)
                t1 = round(int(window[-1]["frame"]) / FPS_FALLBACK, 3)

                desc = _summarize_window(window, CONF_TH, FPS_FALLBACK)

                out.write(json.dumps({
                    "vid": vid,
                    "t0": t0,
                    "t1": t1,
                    "description": desc
                }, ensure_ascii=False) + "\n")

                wrote += 1
                i = j

    print(f"✅ Scene Description: {wrote} đoạn → {SCENE_DESC}")

# ==================== MAIN EXECUTION ====================
if __name__ == "__main__":
    print("🚀 Bắt đầu xử lý...")

    # Bước 1: Convert CSV -> scene_cache
    csv_to_scene_cache()

    # Bước 2: Tạo scene descriptions với spatial-temporal
    build_scene_descriptions()

    # Preview kết quả
    print("\n📄 Preview scene_description.jsonl:")
    with open(SCENE_DESC, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            rec = json.loads(line)
            print(f"  [{rec['vid']}] {rec['t0']:.1f}s-{rec['t1']:.1f}s: {rec['description']}")

✅ Đã load 360 labels từ /content/data/label.json
🚀 Bắt đầu xử lý...
✅ Wrote 742 frames -> roadbuddy/runtime/scene_cache.jsonl
📊 Loaded 742 frames từ 174 videos
✅ Scene Description: 174 đoạn → roadbuddy/runtime/scene_description.jsonl

📄 Preview scene_description.jsonl:
  [01f7c7ec_1000_clip_006_0040_0050_Y] 0.0s-9.0s: Biển: Chỉ Dẫn Chữ×1, Biển Tốc Độ Chữ×1. [ST: Chuyển động TB 163.6px/s]. [2 đối tượng di chuyển].
  [0577aea9_865_clip_021_0161_0166_Y] 2.0s-4.0s: [ST: Chuyển động TB 0.5px/s].
  [0758e201_800_clip_014_0102_0109_Y] 1.0s-3.0s: Biển: Vạch Đứt Vàng×1. [ST: Chuyển động TB 51.8px/s]. [1 đối tượng di chuyển].


In [6]:
# ==== M2.2: VLM Instruction ====
VLM_SYSTEM_PROMPT = (
  "Bạn là trợ lý lái xe RoadBuddy. Trả lời ngắn gọn, chắc chắn, ưu tiên an toàn "
  "và tuân thủ Luật Giao thông đường bộ Việt Nam. Khi đèn/biển/vạch xung đột, "
  "ưu tiên đèn > biển > vạch. Khi không đủ thông tin, hãy nói 'chưa đủ thông tin' "
  "và khuyên 'giảm tốc/giữ làn'. Tránh suy đoán."
)

def build_user_prompt(question: str, scene_text: str = None):
    ctx = f"Cảnh tóm tắt: {scene_text}\n" if scene_text else ""
    return (
        f"{ctx}"
        f"Câu hỏi: {question}\n"
        f"Hãy trả lời ngắn, nêu kết luận chính và (nếu có) căn cứ luật rút gọn."
    )

print("✅ VLM system prompt & helper loaded.")


✅ VLM system prompt & helper loaded.


In [7]:
import torch
from PIL import Image
from transformers import AutoProcessor, InstructBlipForConditionalGeneration

# ưu tiên decord, fallback OpenCV
_USE_DECORD = False
try:
    import decord
    decord.bridge.set_bridge("native")
    _USE_DECORD = True
    print("Sampler: decord")
except Exception:
    import cv2
    print("Sampler: OpenCV fallback")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VLM_MODEL = "Salesforce/instructblip-vicuna-7b"  # GPU yếu: "Salesforce/instructblip-flan-t5-xl"

_VLM = None
_VLM_PROC = None

def _vlm_init(model_name=VLM_MODEL):
    global _VLM, _VLM_PROC
    if _VLM is None:
        _VLM_PROC = AutoProcessor.from_pretrained(model_name)
        _VLM = InstructBlipForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype=(torch.bfloat16 if DEVICE=="cuda" else torch.float32),
        ).to(DEVICE)
    return _VLM, _VLM_PROC

def _sample_frames(video_path: str, max_frames=8):
    if _USE_DECORD:
        import decord
        vr = decord.VideoReader(video_path)
        n = len(vr)
        if n == 0: raise ValueError("Video rỗng.")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        return [Image.fromarray(vr[i].asnumpy()) for i in idxs]
    else:
        import cv2
        cap = cv2.VideoCapture(video_path)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if n <= 0: raise ValueError("Không đọc được video.")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        imgs=[]
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, fr = cap.read()
            if ok:
                fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
                imgs.append(Image.fromarray(fr))
        cap.release()
        if not imgs: raise ValueError("Không lấy được khung hình.")
        return imgs

def vlm_answer(video_path: str, user_prompt: str, system_prompt: str = None,
               frames=8, max_new_tokens=180):
    model, proc = _vlm_init()
    images = _sample_frames(video_path, max_frames=frames)
    prompt = f"[System]\n{system_prompt}\n[User]\n{user_prompt}" if system_prompt else user_prompt
    inputs = proc(images=images, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return proc.batch_decode(out, skip_special_tokens=True)[0]

# Định nghĩa prompt hệ thống để tránh NameError
VLM_SYSTEM_PROMPT = (
  "Bạn là trợ lý lái xe RoadBuddy. Trả lời ngắn gọn, chắc chắn, ưu tiên an toàn "
  "và tuân thủ Luật GTĐB Việt Nam. Khi đèn/biển/vạch xung đột: đèn > biển > vạch. "
  "Nếu không đủ thông tin, hãy nói 'chưa đủ thông tin' và khuyên 'giảm tốc/giữ làn'."
)

print("✅ VLM inference ready: gọi vlm_answer(video_path, user_prompt, VLM_SYSTEM_PROMPT)")


Sampler: OpenCV fallback
✅ VLM inference ready: gọi vlm_answer(video_path, user_prompt, VLM_SYSTEM_PROMPT)


In [8]:
# -------------------- IMPORTS & CONFIG --------------------
import os, re, json, shutil
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

# PdfReader + fallback
try:
    from pypdf import PdfReader
except Exception:
    from PyPDF2 import PdfReader  # fallback

# Dự án
RB_ROOT   = "roadbuddy"
KB_DIR    = f"{RB_ROOT}/kb"
PDF_DIR   = f"{KB_DIR}/pdfs"      # nơi chứa PDF luật
RAG_OUT   = f"{KB_DIR}/index"     # nơi lưu index/chunks/embeddings

# File PDF NGUỒN (đổi theo bạn)
LAW_NEW_SRC = "/content/luat/luatgiaothong.pdf"          # luật mới (bạn đã có)
LAW_OLD_SRC = "/content/luat/luatbosung.pdf"     # nếu có luật cũ; nếu chưa có, vẫn chạy được với 1 file

# Embedding model
EMB_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
CHUNKS_JSONL = f"{RAG_OUT}/chunks.jsonl"
FAISS_INDEX  = f"{RAG_OUT}/faiss.index"

# FAISS optional
try:
    import faiss  # type: ignore
    _FAISS_OK = True
except Exception:
    faiss = None
    _FAISS_OK = False
    print("ℹ️ FAISS không khả dụng → sẽ dùng numpy cosine fallback.")

# -------------------- UTILITIES --------------------
def _pdf_text(path):
    reader = PdfReader(path); txt=[]
    for p in reader.pages:
        t = (p.extract_text() or "").replace("\t"," ")
        t = re.sub(r"[ \u00A0]+"," ", t).strip()
        if t: txt.append(t)
    return "\n".join(txt)

def _chunk(text, max_tokens=220, overlap=40):
    ws = text.split(); out=[]; i=0
    while i < len(ws):
        j = min(i+max_tokens, len(ws))
        out.append(" ".join(ws[i:j]))
        i = j - overlap if j-overlap>i else j
    return out

# -------------------- COPY PDF VỀ THƯ MỤC DỰ ÁN --------------------
os.makedirs(PDF_DIR, exist_ok=True)
pdf_list = []
for src in [LAW_NEW_SRC, LAW_OLD_SRC]:
    if src and os.path.isfile(src):
        dst = f"{PDF_DIR}/{os.path.basename(src)}"
        if os.path.abspath(src) != os.path.abspath(dst):
            shutil.copy(src, dst)
        pdf_list.append(dst)
# Nếu không có luật cũ -> dùng 1 file
if not pdf_list:
    raise FileNotFoundError("Không tìm thấy PDF nào. Hãy chỉnh LAW_NEW_SRC/LAW_OLD_SRC.")
print("PDFs in project:", [Path(p).name for p in pdf_list])

# ƯU TIÊN NGUỒN (file mới ưu tiên điểm cao hơn một chút)
from pathlib import Path as _P
PRIORITY_WEIGHTS = {
    _P(pdf_list[0]).name: 0.12,   # file 1 (thường là luật mới)
}
if len(pdf_list) > 1:
    PRIORITY_WEIGHTS[_P(pdf_list[1]).name] = 0.00  # file 2 (luật cũ)

# -------------------- BUILD INDEX --------------------
def build_rag_index(pdf_paths):
    Path(RAG_OUT).mkdir(parents=True, exist_ok=True)
    corpus=[]
    for p in pdf_paths:
        if not Path(p).exists():
            print(f"⚠️ Không thấy PDF: {p}");
            continue
        raw = _pdf_text(p)
        parts = re.split(r"\n\s*(Điều\s+\d+[^:\n]*:?)", raw, flags=re.I)
        if len(parts)>1:
            for head, body in zip(parts[1::2], parts[2::2]):
                for ch in _chunk(head+"\n"+body):
                    corpus.append({"pdf": Path(p).name, "section": head.strip(), "text": ch})
        else:
            for ch in _chunk(raw):
                corpus.append({"pdf": Path(p).name, "section":"", "text": ch})
    if not corpus:
        raise RuntimeError("Không có dữ liệu để index.")

    model = SentenceTransformer(EMB_MODEL)
    emb = model.encode([c["text"] for c in corpus],
                       convert_to_numpy=True, normalize_embeddings=True,
                       show_progress_bar=True)

    # lưu corpus
    with open(CHUNKS_JSONL,"w",encoding="utf-8") as f:
        for c in corpus: f.write(json.dumps(c, ensure_ascii=False)+"\n")

    # lưu index / embeddings
    if _FAISS_OK:
        index = faiss.IndexFlatIP(emb.shape[1])  # cosine (đã normalize)
        index.add(emb)
        faiss.write_index(index, FAISS_INDEX)
        np.save(f"{RAG_OUT}/embeddings.npy", emb)  # optional
        print(f"✅ RAG indexed {len(corpus)} chunks → {FAISS_INDEX}")
    else:
        np.save(f"{RAG_OUT}/embeddings.npy", emb)
        print(f"✅ Saved embeddings (fallback) {emb.shape} → {RAG_OUT}/embeddings.npy")
    print(f"✅ Saved chunks → {CHUNKS_JSONL}")

# -------------------- SEARCH (ƯU TIÊN + ĐA DẠNG NGUỒN) --------------------
from collections import defaultdict

def _diversify_rank(items, k=5, per_source_cap=3):
    by_src = defaultdict(list)
    for it in items:
        by_src[it["pdf"]].append(it)
    # sort each source
    for s in by_src:
        by_src[s].sort(key=lambda x: x["score"], reverse=True)

    out, used = [], set()
    while len(out) < k:
        progressed = False
        for s in list(by_src.keys()):
            if len(out) >= k: break
            if not by_src[s]: continue
            # cap theo nguồn
            if sum(1 for x in out if x["pdf"]==s) >= per_source_cap:
                continue
            cand = by_src[s].pop(0)
            key = (cand["pdf"], cand["section"], cand["text"][:80])
            if key in used:
                continue
            out.append(cand); used.add(key); progressed = True
        if not progressed:
            break
    # nếu thiếu → bốc theo score
    if len(out) < k:
        pool = [it for lst in by_src.values() for it in lst]
        pool.sort(key=lambda x: x["score"], reverse=True)
        for cand in pool:
            if len(out) >= k: break
            key = (cand["pdf"], cand["section"], cand["text"][:80])
            if key in used: continue
            out.append(cand); used.add(key)
    return out[:k]

def rag_search_two_laws(query, k=5, diversify=True):
    chunks=[json.loads(x) for x in Path(CHUNKS_JSONL).read_text(encoding="utf-8").splitlines()]
    model = SentenceTransformer(EMB_MODEL)
    q = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    if _FAISS_OK and Path(FAISS_INDEX).exists():
        K_RAW = max(k*6, 20)
        index = faiss.read_index(FAISS_INDEX)
        D, I = index.search(q, K_RAW)
        idxs, scores = I[0], D[0]
        items=[]
        for idx, sc in zip(idxs, scores):
            if idx < 0: continue
            c = chunks[int(idx)].copy()
            boost = float(PRIORITY_WEIGHTS.get(c["pdf"], 0.0))
            c["score"] = float(sc) + boost
            items.append(c)
        items.sort(key=lambda x: x["score"], reverse=True)
        return _diversify_rank(items, k=k) if diversify else items[:k]
    else:
        emb = np.load(f"{RAG_OUT}/embeddings.npy")
        raw_scores = (q @ emb.T)[0]  # cosine
        items=[]
        for i, sc in enumerate(raw_scores):
            c = chunks[i].copy()
            boost = float(PRIORITY_WEIGHTS.get(c["pdf"], 0.0))
            c["score"] = float(sc) + boost
            items.append(c)
        items.sort(key=lambda x: x["score"], reverse=True)
        return _diversify_rank(items, k=k) if diversify else items[:k]

# -------------------- BUILD & TEST --------------------
build_rag_index(pdf_list)

query = "tốc độ tối đa trong khu vực đô thị khi có biển tốc độ"
hits = rag_search_two_laws(query, k=4)
for i,h in enumerate(hits,1):
    print(f"\n[{i}] [{h['pdf']}] {h['section']}  score={h['score']:.3f}\n{h['text'][:300]}...")

PDFs in project: ['luatgiaothong.pdf', 'luatbosung.pdf']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

✅ RAG indexed 841 chunks → roadbuddy/kb/index/faiss.index
✅ Saved chunks → roadbuddy/kb/index/chunks.jsonl

[1] [luatgiaothong.pdf] Điều 15. Chuyển hướng xe  score=0.893
đoạn đường cong tầm nhìn bị che khuất, trên đường cao tốc, trong hầm đường bộ, trên đường một chiều, trừ khi có hiệu lệnh của người điều khiển giao thông hoặc chỉ dẫn của biển báo hiệu tạm thời....

[2] [luatbosung.pdf] Điều 35. Tác dụng của biển chỉ dẫn  score=0.785
Điều 35. Tác dụng của biển chỉ dẫn Các biển chỉ dẫn để chỉ dẫn hướng đi hoặc các điều cần thiết nhằm giúp người tham gia giao thông trong việc điều khiển phương tiện và hướng dẫn giao thông trên đường được thuận lợi, đảm bảo an toàn....

[3] [luatgiaothong.pdf] Điều 24. Giao thông tại đường ngang, cầu chung đường sắt  score=0.874
khỏi phạm vi an toàn đường sắt, người điều khiển phương tiện tham gia giao thông đường bộ và người có mặt phải ngay lập tức báo hiệu để dừng tàu, thực hiện các biện pháp bảo đảm an toàn....

[4] [luatbosung.pdf] Điều 32. Ý nghĩa s

In [9]:
# ===================== Cell 2.9 — VLM Context Embedding (LITE, CLIP) =====================
# Nhẹ hơn: dùng CLIP ViT-B/32 để sinh embedding khung hình (không cần Q-Former).
# Yêu cầu: transformers, PIL; decord (nếu có) hoặc OpenCV fallback.

import json, math
from pathlib import Path
from typing import List
import numpy as np
import torch
from PIL import Image

# -------- Cấu hình --------
SCENE_DESC_PATH = "roadbuddy/runtime/scene_description.jsonl"   # input từ M2.1
EMB_DIR         = "roadbuddy/runtime/vlm_ctx"                    # nơi lưu nhúng
CLIP_MODEL      = "openai/clip-vit-base-patch32"                # rất nhẹ, phổ biến
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
FRAMES_PER_WIN  = 6                                             # số frame mẫu / cửa sổ
VIDEO_DIR       = "/content/video"                              # thư mục chứa .mp4
POOLING         = "mean"                                        # 'mean' (gợi ý)

# -------- Model/processor --------
from transformers import AutoProcessor, CLIPModel
_clip_proc  = AutoProcessor.from_pretrained(CLIP_MODEL)
_clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()

# -------- Sampler khung hình --------
try:
    import decord
    decord.bridge.set_bridge("native")
    _USE_DECORD = True
except Exception:
    import cv2
    _USE_DECORD = False

def _sample_frames_from_video(video_path: str, max_frames: int = 6) -> List[Image.Image]:
    if _USE_DECORD:
        vr = decord.VideoReader(video_path)
        n = len(vr)
        if n <= 0: raise ValueError(f"Không đọc được video: {video_path}")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        return [Image.fromarray(vr[i].asnumpy()) for i in idxs]
    else:
        import cv2
        cap = cv2.VideoCapture(video_path)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if n <= 0:
            cap.release()
            raise ValueError(f"Không đọc được video: {video_path}")
        idxs = [int(round(i*(n-1)/(max_frames-1))) for i in range(max_frames)] if max_frames>1 else [0]
        imgs=[]
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, fr = cap.read()
            if ok:
                fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
                imgs.append(Image.fromarray(fr))
        cap.release()
        if not imgs: raise ValueError("Không lấy được khung hình.")
        return imgs

# -------- Hàm tạo embedding cho list ảnh (CLIP) --------
@torch.no_grad()
def clip_embed(images: List[Image.Image]) -> torch.Tensor:
    """
    Trả về vector (dim,) — mean-pool các frame sau khi lấy get_image_features.
    """
    inputs = _clip_proc(images=images, return_tensors="pt").to(DEVICE)
    feats  = _clip_model.get_image_features(pixel_values=inputs["pixel_values"])  # (bs, dim)
    feats  = torch.nn.functional.normalize(feats, dim=-1)
    if POOLING == "mean":
        v = feats.mean(dim=0)                # (dim,)
    else:
        v = feats[0]                         # frame đầu
    v = torch.nn.functional.normalize(v, dim=0)
    return v.detach().float().cpu()

# -------- Duyệt scene_description và lưu nhúng --------
Path(EMB_DIR).mkdir(parents=True, exist_ok=True)
meta_out = []
with open(SCENE_DESC_PATH, "r", encoding="utf-8") as f:
    windows = [json.loads(x) for x in f]

for w in windows:
    vid   = w["vid"]
    vpath = f"{VIDEO_DIR}/{vid}.mp4"   # nếu đuôi khác, đổi theo dữ liệu của bạn
    if not Path(vpath).exists():
        # thử các đuôi phổ biến khác nếu cần
        alt = None
        for ext in [".MP4",".mov",".MOV",".mkv",".MKV",".avi",".AVI"]:
            if Path(f"{VIDEO_DIR}/{vid}{ext}").exists():
                alt = f"{VIDEO_DIR}/{vid}{ext}"; break
        if not alt:
            # không tìm thấy video -> bỏ qua
            continue
        vpath = alt

    try:
        imgs = _sample_frames_from_video(vpath, max_frames=FRAMES_PER_WIN)
        vec  = clip_embed(imgs)                           # (dim,)
        npy_path = f"{EMB_DIR}/{vid}_{w['t0']:.3f}_{w['t1']:.3f}.npy"
        np.save(npy_path, vec.numpy())
        meta_out.append({"vid": vid, "t0": w["t0"], "t1": w["t1"], "embed_path": npy_path, "dim": int(vec.numel())})
    except Exception as e:
        print(f"[WARN] {vid} [{w['t0']}-{w['t1']}] lỗi nhúng: {e}")

with open(f"{EMB_DIR}/vlm_ctx_meta.jsonl", "w", encoding="utf-8") as f:
    for m in meta_out:
        f.write(json.dumps(m, ensure_ascii=False) + "\n")

print(f"✅ Saved {len(meta_out)} CLIP embeddings → {EMB_DIR}")
print("Ví dụ 1 dòng meta:", meta_out[0] if meta_out else "(trống)")


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

✅ Saved 0 CLIP embeddings → roadbuddy/runtime/vlm_ctx
Ví dụ 1 dòng meta: (trống)


In [10]:
# ============== M2.T — Temporal Facts per window ==============
import json, math
from collections import Counter, defaultdict
from pathlib import Path

SCENE_CACHE_PATH = "roadbuddy/runtime/scene_cache.jsonl"
SCENE_DESC_PATH  = "roadbuddy/runtime/scene_description.jsonl"
TEMP_FACTS_PATH  = "roadbuddy/runtime/temporal_facts.jsonl"
FPS, CONF_TH = 15, 0.35

def _iter_jsonl(p):
    with open(p,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: yield json.loads(line)

def _range_frames(t0, t1, fps=FPS):
    return range(int(math.floor(t0*fps)), int(math.ceil(t1*fps))+1)

def build_temporal_facts():
    if not Path(SCENE_CACHE_PATH).exists() or not Path(SCENE_DESC_PATH).exists():
        raise FileNotFoundError("Thiếu scene_cache hoặc scene_description.")
    by_vid = defaultdict(dict)
    for r in _iter_jsonl(SCENE_CACHE_PATH):
        by_vid[r["vid"]][int(r["frame"])] = r

    wrote=0
    with open(TEMP_FACTS_PATH,"w",encoding="utf-8") as out:
        for w in _iter_jsonl(SCENE_DESC_PATH):
            vid, t0, t1 = w["vid"], float(w["t0"]), float(w["t1"])
            frames = _range_frames(t0, t1, FPS)
            light_hist, sign_hist, lane_hist, ocr_hist = Counter(), Counter(), Counter(), Counter()
            consec = {"red":0,"yellow":0,"green":0}
            max_consec = {"red":0,"yellow":0,"green":0}

            for f in frames:
                r = by_vid.get(vid,{}).get(f)
                if not r:
                    for k in consec: consec[k]=0
                    continue
                # lights
                frame_has=set()
                for l in r.get("lights",[]):
                    if float(l.get("conf",0))>=CONF_TH:
                        cls=int(l.get("cls",-1))
                        name={0:"red",1:"yellow",2:"green",3:"green_left",4:"green_straight"}.get(cls,f"light_{cls}")
                        light_hist[name]+=1
                        if name in ("red","yellow","green"): frame_has.add(name)
                for k in consec:
                    if k in frame_has:
                        consec[k]+=1; max_consec[k]=max(max_consec[k],consec[k])
                    else:
                        consec[k]=0
                # signs / lanes / ocr
                for s in r.get("signs",[]):
                    if float(s.get("conf",0))>=CONF_TH:
                        sign_hist[str(s.get("name") or f"sign_{s.get('cls',-1)}")]+=1
                for ln in r.get("lanes",[]):
                    t=ln.get("type");
                    if t: lane_hist[t]+=1
                for o in r.get("ocr",[]):
                    txt=str(o.get("text","")).strip()
                    if txt: ocr_hist[txt]+=1

            dur=lambda c: round(c/float(FPS),3)
            facts={
                "vid":vid,"t0":t0,"t1":t1,
                "light_dominant": (light_hist.most_common(1)[0][0] if light_hist else "unknown"),
                "red_stable_sec": dur(max_consec["red"]),
                "green_stable_sec": dur(max_consec["green"]),
                "yellow_stable_sec": dur(max_consec["yellow"]),
                "sign_top":[k for k,_ in sign_hist.most_common(5)],
                "lane_top":[k for k,_ in lane_hist.most_common(4)],
                "ocr_top":[k for k,_ in ocr_hist.most_common(6)],
            }
            out.write(json.dumps(facts,ensure_ascii=False)+"\n"); wrote+=1
    print(f"✅ Temporal facts: {wrote} đoạn → {TEMP_FACTS_PATH}")

# chạy 1 lần sau khi có scene_description.jsonl
build_temporal_facts()


✅ Temporal facts: 174 đoạn → roadbuddy/runtime/temporal_facts.jsonl


Module3


In [11]:
# ==================== CELL 1: ONE-TIME INSTALL ====================
!pip -q install --upgrade transformers accelerate bitsandbytes
!pip -q install --upgrade opencv-python easyocr
!pip -q install --upgrade sentence-transformers faiss-cpu

import importlib
def _ok(pkg, alias=None):
    try:
        importlib.import_module(alias or pkg)
        print(f"✅ {pkg} OK")
    except Exception as e:
        print(f"⚠️ {pkg} không sẵn sàng:", e)

_ok("transformers")
_ok("accelerate")
_ok("bitsandbytes")
_ok("opencv-python","cv2")
_ok("easyocr")
_ok("sentence_transformers")
_ok("faiss")
print("✅ All dependencies OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 26.9 MB/s eta 0:00:00
✅ transformers OK
✅ accelerate OK
✅ bitsandbytes OK
✅ opencv-python OK
✅ easyocr OK
✅ sentence_transformers OK
✅ faiss OK
✅ All dependencies OK


In [12]:
# ==================== CELL 2: CONFIG & PATHS ====================
from pathlib import Path

QUESTIONS_JSON = "/content/data/public_test.json"
META_JSON      = "/content/runs_meta/metadata.json"
NORM_JSON      = "/content/data/label.json"
VIDEO_ROOT     = "/content/drive/MyDrive/videos"

# OUTPUT
OUT_SHORT = "roadbuddy/runtime/answers_short.csv"
OUT_FULL  = "roadbuddy/runtime/answers_full.csv"

# SCENE / TEMPORAL / RAG
SCENE_DESC_PATH = "roadbuddy/runtime/scene_description.jsonl"
TEMPORAL_PATH   = "roadbuddy/runtime/temporal_facts.jsonl"
FAISS_INDEX_PATH = "roadbuddy/kb/index/faiss.index"
CHUNKS_JSONL     = "roadbuddy/kb/index/chunks.jsonl"

RAG_TOPK = 3
OCR_MAX_FRAMES = 6

PHI_MODEL = "microsoft/Phi-3-mini-4k-instruct"
USE_PHI   = True
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Ensure output folder exists
Path(Path(OUT_SHORT).parent).mkdir(parents=True, exist_ok=True)
Path(Path(OUT_FULL).parent).mkdir(parents=True, exist_ok=True)

print("✅ Config OK")



✅ Config OK


In [13]:
# ==================== CELL 3: IMPORTS ====================
import os, re, json, time, csv, statistics
from pathlib import Path
from typing import List, Dict, Any, Optional
from collections import Counter, defaultdict

import numpy as np
import cv2
import easyocr
import faiss

import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

print("✅ Imports OK")

✅ Imports OK


In [14]:
# ==================== CELL 4: MEMORY CLEANUP ====================
# Xóa các model không cần thiết
try:
    del _VLM, _VLM_PROC
    print("✅ Deleted VLM model")
except:
    pass

try:
    del ocr_reader
    print("✅ Deleted old OCR reader")
except:
    pass

# Giải phóng CUDA cache
torch.cuda.empty_cache()
gc.collect()

# Kiểm tra memory
if torch.cuda.is_available():
    print(f"📊 GPU Memory:")
    print(f"   Allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
    print(f"   Reserved:  {torch.cuda.memory_reserved(0)/1e9:.2f} GB")
    total = torch.cuda.get_device_properties(0).total_memory
    free = total - torch.cuda.memory_reserved(0)
    print(f"   Free:      {free/1e9:.2f} GB / {total/1e9:.2f} GB")

print("✅ Memory cleanup done")

✅ Deleted VLM model
📊 GPU Memory:
   Allocated: 0.62 GB
   Reserved:  0.65 GB
   Free:      15.18 GB / 15.83 GB
✅ Memory cleanup done


In [15]:
# ==================== CELL 5: LOADERS ====================
def load_questions(path):
    """Load questions từ file JSON với cấu trúc {__count__, data: [...]}"""
    txt = Path(path).read_text(encoding="utf-8")

    try:
        data = json.loads(txt)
    except Exception as e:
        raise RuntimeError(f"❌ Không parse được file JSON: {e}")

    # Kiểm tra cấu trúc {__count__, data}
    if isinstance(data, dict):
        if "data" in data:
            records = data["data"]
            print(f"✅ Detected structure: {{__count__: {data.get('__count__')}, data: [...]}}")
        else:
            records = [data]
    elif isinstance(data, list):
        records = data
    else:
        raise RuntimeError(f"❌ Định dạng JSON không hợp lệ: {type(data)}")

    # Parse từng record
    out = []
    for i, r in enumerate(records):
        if isinstance(r, str):
            if r in ["__count__", "data"]:
                continue
            try:
                r = json.loads(r)
            except Exception as e:
                print(f"⚠️ Không parse được record {i}: {r[:100]}")
                continue

        if not isinstance(r, dict):
            continue

        qid = str(r.get("id", ""))
        question = r.get("question") or r.get("q", "")
        options = r.get("options") or r.get("opts") or r.get("choices") or []
        video = r.get("video") or r.get("video_path", "")

        if not (qid and question and video):
            continue

        out.append({
            "id": qid,
            "question": question,
            "options": options,
            "video": video
        })

    print(f"✅ Loaded {len(out)} câu hỏi từ {path}")
    if out:
        print(f"   Ví dụ: id={out[0]['id']}, q={out[0]['question'][:50]}...")

    return out


def _norm_key(p: str) -> str:
    return Path(p).name.strip()


def load_meta(path: str):
    mp = {}
    if not Path(path).exists():
        print(f"⚠️ Không tìm thấy meta: {path}")
        return mp
    raw = json.loads(Path(path).read_text())
    for it in raw.get("items", []):
        k = _norm_key(it.get("video_path",""))
        objs=[]
        for kf in it.get("keyframes", []):
            for o in kf.get("objects", []):
                objs.append({"name":o["name"],"cls":o["cls"],"conf":o["conf"]})
        mp[k]=objs
    print(f"✅ Loaded meta: {len(mp)} videos")
    return mp


def load_norm(path: str):
    if not Path(path).exists():
        print(f"⚠️ Không tìm thấy norm: {path}")
        return {}
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    result = {k.lower():v for k,v in raw.items()}
    print(f"✅ Loaded norm: {len(result)} mappings")
    return result


def load_scene_desc_map(path=SCENE_DESC_PATH):
    mp={}
    if not Path(path).exists():
        print(f"⚠️ Không tìm thấy scene desc: {path}")
        return mp
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            r=json.loads(ln)
            k=_norm_key(r["vid"])
            mp.setdefault(k,[]).append(r)
    print(f"✅ Loaded scene: {len(mp)} videos")
    return mp


def load_temporal_map(path=TEMPORAL_PATH):
    mp={}
    if not Path(path).exists():
        print(f"⚠️ Không tìm thấy temporal: {path}")
        return mp
    with open(path,"r",encoding="utf-8") as f:
        for ln in f:
            r=json.loads(ln)
            k=_norm_key(r["vid"])
            mp.setdefault(k,[]).append(r)
    print(f"✅ Loaded temporal: {len(mp)} videos")
    return mp

print("✅ Loaders OK")


✅ Loaders OK


In [16]:
# ==================== CELL 6: NORMALIZE META ====================
def normalize_names(raw_objs, norm_map, min_conf=0.15):
    out=[]
    for o in raw_objs:
        if float(o["conf"]) < min_conf: continue
        base = o["name"].strip()
        vn   = norm_map.get(base.lower(), base)
        out.append(vn)
    return out[:12]

print("✅ Normalize OK")

✅ Normalize OK


In [17]:
# ==================== CELL 7: OCR ====================
ocr_reader = easyocr.Reader(['vi','en'], gpu=torch.cuda.is_available())

def sample_frames(video_path, max_frames=OCR_MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return []
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(n-1,1), max_frames, dtype=int)
    frames=[]
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, fr = cap.read()
        if ok: frames.append(fr)
    cap.release()
    return frames

def ocr_video_texts(video_path):
    if not Path(video_path).exists():
        alt = str(Path(VIDEO_ROOT)/Path(video_path).name)
        if Path(alt).exists(): video_path=alt
        else: return []

    texts=[]
    for fr in sample_frames(video_path):
        rgb = fr[:,:,::-1]
        res = ocr_reader.readtext(rgb, detail=0)
        for t in res:
            t = re.sub(r"\s+"," ",t).strip()
            if len(t)>=2 and t not in texts:
                texts.append(t)
    return texts[:20]

print("✅ OCR OK")



Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ OCR OK


In [18]:
# ==================== CELL 8: RAG FAISS ====================
_emb_model = None

def _get_embedding_model():
    global _emb_model
    if _emb_model is None:
        _emb_model = SentenceTransformer(EMB_MODEL)
        print(f"✅ Loaded embedding model: {EMB_MODEL}")
    return _emb_model

# Load FAISS index
rag_index = None
if Path(FAISS_INDEX_PATH).exists():
    try:
        rag_index = faiss.read_index(FAISS_INDEX_PATH)
        print(f"✅ Loaded FAISS: {rag_index.ntotal} vectors")
    except Exception as e:
        print(f"⚠️ Cannot load FAISS: {e}")

# Load chunks
rag_chunks = []
if Path(CHUNKS_JSONL).exists():
    with open(CHUNKS_JSONL, "r", encoding="utf-8") as f:
        for ln in f:
            rag_chunks.append(json.loads(ln))
    print(f"✅ Loaded {len(rag_chunks)} chunks")

def rag_fetch(query: str, k=RAG_TOPK):
    """Search RAG với embedding thật"""
    if rag_index is None or not rag_chunks:
        return []

    try:
        model = _get_embedding_model()
        qvec = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        D, I = rag_index.search(qvec.astype('float32'), k)

        results = []
        for idx, score in zip(I[0], D[0]):
            if idx < len(rag_chunks) and idx >= 0:
                chunk = rag_chunks[idx]
                text = f"[{chunk.get('pdf', 'unknown')}] {chunk.get('section', '')}: {chunk['text'][:200]}"
                results.append(text)
        return results
    except Exception as e:
        print(f"⚠️ RAG error: {e}")
        return []

print("✅ RAG OK")

✅ Loaded FAISS: 841 vectors
✅ Loaded 841 chunks
✅ RAG OK


In [19]:
# ==================== CELL 9: SCENE & TEMPORAL ====================
def scene_text_for_video(vkey, scene_map, limit=3):
    snaps = scene_map.get(vkey, [])
    snaps = sorted(snaps, key=lambda x:x.get("t0",0))[:limit]
    return " ".join(f"[{s['t0']}-{s['t1']}s] {s['description']}" for s in snaps)

def rule_engine_temporal(vkey, temporal_map):
    ws = temporal_map.get(vkey, [])
    if not ws: return []
    w = sorted(ws, key=lambda x:x.get("t0",0))[0]
    adv=[]
    if w.get("red_stable_sec",0)>=2: adv.append("Đèn đỏ ≥2s: dừng")
    if w.get("green_stable_sec",0)>=2: adv.append("Đèn xanh: đi")
    if w.get("yellow_stable_sec",0)>=1: adv.append("Đèn vàng: dừng")
    txt = " ".join(w.get("ocr_top", []))
    m = re.findall(r"\b(\d{2,3})\b", txt)
    sp = [int(x) for x in m if 15<=int(x)<=120]
    if sp: adv.append(f"Tốc độ max: {max(sp)} km/h")
    return adv

print("✅ Scene/Temporal OK")




✅ Scene/Temporal OK


In [20]:
# ==================== CELL 10: PHI-3 MODEL - LOAD & INFERENCE ====================

# ===== PHẦN 1: LOAD MODEL VỚI AUTO-FALLBACK =====
_USE_PHI_OK = False
tok = None
phi = None

if USE_PHI:
    print("🔄 Loading Phi-3 model...")

    try:
        # ===== TRY 1: 4-BIT QUANTIZATION (BEST) =====
        try:
            print("   Attempting 4-bit quantization...")
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4"
            )

            tok = AutoTokenizer.from_pretrained(PHI_MODEL, trust_remote_code=True)
            phi = AutoModelForCausalLM.from_pretrained(
                PHI_MODEL,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                low_cpu_mem_usage=True,
                torch_dtype=torch.float16
            )
            _USE_PHI_OK = True
            print("✅ Phi-3 loaded successfully (4-bit)")

        except Exception as e4:
            print(f"⚠️ 4-bit failed: {str(e4)[:100]}")
            print("🔄 Trying 8-bit quantization...")

            # ===== TRY 2: 8-BIT QUANTIZATION (FALLBACK 1) =====
            try:
                bnb_config = BitsAndBytesConfig(
                    load_in_8bit=True,
                    llm_int8_threshold=6.0,
                    llm_int8_has_fp16_weight=False
                )

                tok = AutoTokenizer.from_pretrained(PHI_MODEL, trust_remote_code=True)
                phi = AutoModelForCausalLM.from_pretrained(
                    PHI_MODEL,
                    quantization_config=bnb_config,
                    device_map="auto",
                    trust_remote_code=True,
                    low_cpu_mem_usage=True
                )
                _USE_PHI_OK = True
                print("✅ Phi-3 loaded successfully (8-bit)")

            except Exception as e8:
                print(f"⚠️ 8-bit failed: {str(e8)[:100]}")
                print("🔄 Trying float16 (no quantization)...")

                # ===== TRY 3: FLOAT16 NO QUANTIZATION (FALLBACK 2) =====
                tok = AutoTokenizer.from_pretrained(PHI_MODEL, trust_remote_code=True)
                phi = AutoModelForCausalLM.from_pretrained(
                    PHI_MODEL,
                    torch_dtype=torch.float16,
                    device_map="auto",
                    trust_remote_code=True,
                    low_cpu_mem_usage=True
                )
                _USE_PHI_OK = True
                print("✅ Phi-3 loaded successfully (float16)")

        # ===== CHECK GPU MEMORY =====
        if torch.cuda.is_available() and _USE_PHI_OK:
            allocated = torch.cuda.memory_allocated(0) / 1e9
            reserved = torch.cuda.memory_reserved(0) / 1e9
            total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"📊 GPU Memory Status:")
            print(f"   Allocated: {allocated:.2f} GB")
            print(f"   Reserved:  {reserved:.2f} GB")
            print(f"   Total:     {total:.2f} GB")
            print(f"   Free:      {total - reserved:.2f} GB")

    except Exception as e:
        print(f"❌ All loading attempts failed: {str(e)[:150]}")
        _USE_PHI_OK = False
        tok = None
        phi = None

else:
    print("⚠️ USE_PHI=False, skipping model load")

print(f"\n{'='*60}")
print(f"🎯 Phi-3 Status: {'✅ READY' if _USE_PHI_OK else '❌ NOT LOADED'}")
print(f"{'='*60}\n")


# ===== PHẦN 2: INFERENCE FUNCTION =====
def infer_phi(q, options, scene, labels, ocr, laws, temp):
    """
    Suy luận với Phi-3 model

    Args:
        q: Câu hỏi
        options: List các đáp án
        scene: Mô tả cảnh
        labels: Nhãn phát hiện
        ocr: Text từ OCR
        laws: Luật giao thông từ RAG
        temp: Temporal facts

    Returns:
        (answer_letter, explanation)
    """

    # ===== CHECK MODEL STATUS =====
    if not _USE_PHI_OK or phi is None or tok is None:
        import random
        fallback = random.choice(['A', 'B', 'C', 'D'])
        return fallback, "⚠️ Model chưa được load"

    try:
        # ===== BUILD CONTEXT =====
        ctx_parts = []

        # Scene (prioritize)
        if scene and scene.strip():
            ctx_parts.append(f"Cảnh: {scene[:150]}")

        # Labels (top 4)
        if labels:
            ctx_parts.append(f"Nhãn: {', '.join(labels[:4])}")

        # Temporal facts (high priority)
        if temp:
            ctx_parts.append(f"Thời gian: {'; '.join(temp[:2])}")

        # Laws (only if relevant)
        if laws:
            law_keywords = ['luật', 'quy định', 'phạt', 'điều', 'km/h', 'tốc độ', 'biển báo']
            if any(kw in q.lower() for kw in law_keywords):
                ctx_parts.append(f"Luật: {laws[0][:100]}")

        # OCR (only numbers)
        if ocr:
            ocr_nums = [x for x in ocr if re.search(r'\d', x)]
            if ocr_nums:
                ctx_parts.append(f"OCR: {', '.join(ocr_nums[:3])}")

        context = " | ".join(ctx_parts) if ctx_parts else "Không có thông tin bổ sung."

        # ===== BUILD PROMPT =====
        opt_txt = "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(options))

        prompt = (
            "Bạn là trợ lý lái xe. Trả lời câu hỏi bằng cách chọn đáp án đúng.\n"
            "Format trả lời:\n"
            "- Dòng đầu: Chỉ ghi 1 chữ cái A, B, C hoặc D\n"
            "- Dòng sau: Lý do (ngắn gọn)\n\n"
            f"Câu hỏi: {q}\n\n"
            f"Các đáp án:\n{opt_txt}\n\n"
            f"Thông tin bổ sung:\n{context}\n\n"
            "Ví dụ:\n"
            "A\n"
            "Theo luật, đèn đỏ phải dừng.\n\n"
            "Trả lời:"
        )

        # ===== TOKENIZE =====
        inp = tok(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=850  # Giảm để tránh OOM
        ).to(phi.device)

        prompt_len = inp.input_ids.shape[1]

        # ===== GENERATE =====
        with torch.no_grad():
            out = phi.generate(
                **inp,
                max_new_tokens=60,
                do_sample=True,
                temperature=0.3,      # Thấp = ổn định
                top_p=0.85,
                top_k=40,
                pad_token_id=tok.eos_token_id,
                use_cache=False       # Tắt cache để tránh DynamicCache bug
            )

        # ===== DECODE =====
        gen_tokens = out[0][prompt_len:]
        response = tok.decode(gen_tokens, skip_special_tokens=True).strip()

        # ===== CLEANUP MEMORY =====
        del out, inp, gen_tokens
        torch.cuda.empty_cache()

        # ===== PARSE RESPONSE (7 PATTERNS) =====
        if response:
            lines = [l.strip() for l in response.split('\n') if l.strip()]

            # Patterns từ chặt → lỏng
            patterns = [
                r"^([ABCD])\s*$",                      # "A"
                r"^([ABCD])\s*[\.:\-\)]",              # "A." "A:" "A-"
                r"^đáp\s*án[:\s]*([ABCD])\b",          # "Đáp án: A"
                r"^chọn[:\s]*([ABCD])\b",              # "Chọn A"
                r"^trả\s*lời[:\s]*([ABCD])\b",         # "Trả lời: A"
                r"^([ABCD])\s+\w+",                    # "A Dừng lại"
                r"\b([ABCD])\b"                        # Fallback
            ]

            # Search in first 3 lines
            for line in lines[:3]:
                for pattern in patterns:
                    match = re.search(pattern, line, re.IGNORECASE)
                    if match:
                        ans_letter = match.group(1).upper()
                        ans_idx = ord(ans_letter) - 65

                        # Validate
                        if 0 <= ans_idx < len(options):
                            # Get explanation
                            expl_lines = [l for l in lines[1:4] if l and not re.match(r'^[ABCD]\s*$', l)]
                            expl = " ".join(expl_lines[:2]) if expl_lines else f"Chọn {ans_letter}"
                            return ans_letter, expl[:200]

        # ===== PARSE FAILED → RETRY WITH SIMPLE PROMPT =====
        print(f"⚠️ Parse failed (response: '{response[:80]}'), retrying...")

        simple_prompt = (
            f"Câu hỏi: {q}\n\n"
            f"{opt_txt}\n\n"
            "Trả lời bằng 1 chữ cái A, B, C hoặc D (chỉ ghi chữ cái):\n"
        )

        inp2 = tok(
            simple_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=600
        ).to(phi.device)

        prompt_len2 = inp2.input_ids.shape[1]

        with torch.no_grad():
            out2 = phi.generate(
                **inp2,
                max_new_tokens=5,
                do_sample=False,  # Greedy
                pad_token_id=tok.eos_token_id,
                use_cache=False
            )

        gen_tokens2 = out2[0][prompt_len2:]
        response2 = tok.decode(gen_tokens2, skip_special_tokens=True).strip()

        del out2, inp2, gen_tokens2
        torch.cuda.empty_cache()

        # Parse retry
        match2 = re.search(r'\b([ABCD])\b', response2, re.IGNORECASE)
        if match2:
            ans2 = match2.group(1).upper()
            if 0 <= ord(ans2) - 65 < len(options):
                return ans2, "⚠️ Retry đơn giản thành công"

        # ===== STILL FAILED → RANDOM =====
        print(f"⚠️ Retry failed (response: '{response2}'), using random...")
        import random
        fallback = random.choice(['A', 'B', 'C', 'D'])
        return fallback, "⚠️ Model không parse được, chọn ngẫu nhiên"

    except torch.cuda.OutOfMemoryError:
        print("❌ GPU Out of Memory!")
        torch.cuda.empty_cache()
        gc.collect()
        import random
        return random.choice(['A', 'B', 'C', 'D']), "❌ OOM - GPU hết bộ nhớ"

    except Exception as e:
        print(f"❌ Inference error: {str(e)[:150]}")
        torch.cuda.empty_cache()
        import random
        return random.choice(['A', 'B', 'C', 'D']), f"❌ Error: {str(e)[:100]}"


print("✅ Phi-3 inference function ready - Sẵn sàng chạy suy luận")

🔄 Loading Phi-3 model...
   Attempting 4-bit quantization...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Phi-3 loaded successfully (float16)
📊 GPU Memory Status:
   Allocated: 8.36 GB
   Reserved:  9.30 GB
   Total:     15.83 GB
   Free:      6.53 GB

🎯 Phi-3 Status: ✅ READY

✅ Phi-3 inference function ready - Sẵn sàng chạy suy luận


In [21]:
# ==================== CELL 11: ANSWER_ONE ====================
def answer_one(r, meta_map, norm_map, scene_map, temporal_map):
    qid   = r["id"]
    qtext = r["question"]
    opts  = r.get("options",[])
    vpath = r["video"]
    vkey  = _norm_key(vpath)

    # Meta
    raw_objs = meta_map.get(vkey, [])
    norm_labels = normalize_names(raw_objs, norm_map)

    # OCR - TẮT để tiết kiệm RAM nếu cần
    video_abs = str(Path(VIDEO_ROOT)/vkey)
    try:
        ocr_list = ocr_video_texts(video_abs)
    except:
        ocr_list = []

    # Scene + temporal
    scene_text = scene_text_for_video(vkey, scene_map) if vkey in scene_map else ""
    temporal_advice = rule_engine_temporal(vkey, temporal_map)

    # RAG
    law_snips = rag_fetch(qtext)

    # Phi
    ansL, expl = infer_phi(qtext, opts, scene_text, norm_labels, ocr_list, law_snips, temporal_advice)

    # Map answer
    idx = ord(ansL)-65
    ansT = opts[idx] if 0<=idx<len(opts) else ""

    return {
        "id":qid,
        "video_path":vpath,
        "question":qtext,
        "options":opts,
        "answer_letter":ansL,
        "answer_text":ansT,
        "explain":expl,
        "scene_text":scene_text,
        "temporal_advice":" | ".join(temporal_advice),
        "norm_labels":"; ".join(norm_labels),
        "ocr_texts":" | ".join(ocr_list),
        "law_refs":" | ".join(law_snips),
    }

print("✅ answer_one OK")


✅ answer_one OK


In [22]:
# ==================== THÊM CELL MỚI: DEBUG MODE ====================
# Chèn vào trước CELL 12 để debug

DEBUG_MODE = True  # Bật để xem chi tiết

def answer_one_debug(r, meta_map, norm_map, scene_map, temporal_map):
    """Version debug - in ra tất cả thông tin"""
    qid = r["id"]
    qtext = r["question"]

    if DEBUG_MODE:
        print(f"\n{'='*60}")
        print(f"🔍 DEBUG Câu {qid}")
        print(f"   Q: {qtext[:80]}...")

    # Gọi answer_one như cũ
    result = answer_one(r, meta_map, norm_map, scene_map, temporal_map)

    if DEBUG_MODE:
        print(f"   → Ans: {result['answer_letter']}")
        print(f"   → Explain: {result['explain'][:100]}...")

        # Kiểm tra context length
        total_len = (
            len(result.get('scene_text', '')) +
            len(result.get('norm_labels', '')) +
            len(result.get('ocr_texts', '')) +
            len(result.get('law_refs', ''))
        )
        print(f"   → Context length: {total_len} chars")

        if total_len > 500:
            print(f"   ⚠️ Context quá dài! (>{500})")

    return result


In [23]:
# ==================== CELL 12: STREAMING RUNNER (FIXED) ====================
def run_module3_streaming(
    questions_json=QUESTIONS_JSON,
    meta_json=META_JSON,
    norm_json=NORM_JSON,
    out_short=OUT_SHORT,
    out_full=OUT_FULL):

    # Load
    qrows = load_questions(questions_json)
    meta_map = load_meta(meta_json)
    norm_map = load_norm(norm_json)
    scene_map = load_scene_desc_map()
    temporal_map = load_temporal_map()

    # Prepare output
    short_p = Path(out_short)
    full_p  = Path(out_full)
    short_p.parent.mkdir(parents=True, exist_ok=True)
    full_p.parent.mkdir(parents=True, exist_ok=True)

    if short_p.exists(): short_p.unlink()
    if full_p.exists(): full_p.unlink()

    full_header=[
        "id","video_path","question","options","answer_letter","answer_text",
        "explain","scene_text","temporal_advice","norm_labels","ocr_texts","law_refs"
    ]

    with open(out_full,"w",encoding="utf-8",newline="") as f:
        csv.DictWriter(f, fieldnames=full_header).writeheader()

    print(f"[RUN] Bắt đầu {len(qrows)} câu...")

    success, error = 0, 0

    for idx, r in enumerate(qrows, 1):
        try:
            out = answer_one(r, meta_map, norm_map, scene_map, temporal_map)

            # Short CSV
            with open(out_short, "a", encoding="utf-8") as f:
                f.write(f"câu{out['id']}:{out['answer_letter']}\n")

            # Full CSV
            row = dict(out)
            row["options"] = " || ".join(row["options"]) if isinstance(row["options"], list) else row["options"]

            with open(out_full, "a", encoding="utf-8", newline="") as f:
                csv.DictWriter(f, fieldnames=full_header).writerow(row)

            success += 1

        except Exception as e:
            error += 1
            print(f"❌ Lỗi câu {r.get('id', idx)}: {e}")

            # FALLBACK THÔNG MINH thay vì luôn A
            import random
            fallback_ans = random.choice(['A', 'B', 'C', 'D'])

            # Ghi vào short CSV
            with open(out_short, "a", encoding="utf-8") as f:
                f.write(f"câu{r.get('id', idx)}:{fallback_ans}\n")

            # Ghi vào full CSV để debug
            with open(out_full, "a", encoding="utf-8", newline="") as f:
                error_row = {
                    "id": r.get('id', idx),
                    "video_path": r.get('video', ''),
                    "question": r.get('question', ''),
                    "options": " || ".join(r.get('options', [])),
                    "answer_letter": fallback_ans,
                    "answer_text": "",
                    "explain": f"❌ Exception: {str(e)[:100]}",
                    "scene_text": "",
                    "temporal_advice": "",
                    "norm_labels": "",
                    "ocr_texts": "",
                    "law_refs": ""
                }
                csv.DictWriter(f, fieldnames=full_header).writerow(error_row)

        # Progress & cleanup (QUAN TRỌNG: Phải nằm ngoài try-except)
        if idx % 10 == 0 or idx == len(qrows):
            print(f"  [{idx}/{len(qrows)}] ✅ {success} | ❌ {error}")
            torch.cuda.empty_cache()

    print(f"\n🎉 HOÀN THÀNH!")
    print(f"   ✅ {success}/{len(qrows)}")
    print(f"   ❌ {error}")
    print(f"   📄 {out_short}")
    print(f"   📄 {out_full}")

    return out_short, out_full

print("✅ Runner OK")

✅ Runner OK


In [ ]:
# ==================== CELL 13: RUN ====================
run_module3_streaming()

✅ Detected structure: {__count__: 405, data: [...]}
✅ Loaded 405 câu hỏi từ /content/data/public_test.json
   Ví dụ: id=testa_0001, q=Theo trong video, nếu ô tô đi hướng chếch sang phả...
✅ Loaded meta: 182 videos
✅ Loaded norm: 360 mappings
✅ Loaded scene: 174 videos
✅ Loaded temporal: 174 videos
[RUN] Bắt đầu 405 câu...
✅ Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


  [10/405] ✅ 10 | ❌ 0
  [20/405] ✅ 20 | ❌ 0
  [30/405] ✅ 30 | ❌ 0
  [40/405] ✅ 40 | ❌ 0
  [50/405] ✅ 50 | ❌ 0
  [60/405] ✅ 60 | ❌ 0
  [70/405] ✅ 70 | ❌ 0
  [80/405] ✅ 80 | ❌ 0
  [90/405] ✅ 90 | ❌ 0
  [100/405] ✅ 100 | ❌ 0
  [110/405] ✅ 110 | ❌ 0
  [120/405] ✅ 120 | ❌ 0
  [130/405] ✅ 130 | ❌ 0
  [140/405] ✅ 140 | ❌ 0
  [150/405] ✅ 150 | ❌ 0
  [160/405] ✅ 160 | ❌ 0
  [170/405] ✅ 170 | ❌ 0
  [180/405] ✅ 180 | ❌ 0
  [190/405] ✅ 190 | ❌ 0
  [200/405] ✅ 200 | ❌ 0
  [210/405] ✅ 210 | ❌ 0
  [220/405] ✅ 220 | ❌ 0
  [230/405] ✅ 230 | ❌ 0
  [240/405] ✅ 240 | ❌ 0
  [250/405] ✅ 250 | ❌ 0
  [260/405] ✅ 260 | ❌ 0
  [270/405] ✅ 270 | ❌ 0
  [280/405] ✅ 280 | ❌ 0
⚠️ Parse failed (response: 'ật điều 73, biển cấm đi ngược chiều, biển cảnh báo đường đôi, biển cảnh báo gậ'), retrying...
  [290/405] ✅ 290 | ❌ 0
  [300/405] ✅ 300 | ❌ 0
  [310/405] ✅ 310 | ❌ 0
  [320/405] ✅ 320 | ❌ 0
  [330/405] ✅ 330 | ❌ 0
  [340/405] ✅ 340 | ❌ 0
  [350/405] ✅ 350 | ❌ 0
  [360/405] ✅ 360 | ❌ 0
  [370/405] ✅ 370 | ❌ 0
  [380/4

In [2]:
# Tìm MỌI file CSV trong /content
!find /content -name "*.csv" -type f 2>/dev/null

# Kiểm tra folder có tồn tại không
!ls -la /content/roadbuddy/ 2>/dev/null
!ls -la /content/roadbuddy/runtime/ 2>/dev/null

# Tìm file backup hoặc temp
!find /content -name "*answer*" -type f 2>/dev/null
!find /content -name "*output*" -type f 2>/dev/null

/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
